In [1]:
import pandas as pd

# Load the dataset
file_path = "crime_data_engineered.csv"   
df = pd.read_csv(file_path)

# Display basic info and preview
print("🛠️ Dataset Info:")
print(df.info())
print("\n Preview:")
print(df.head())


🛠️ Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 272264 entries, 0 to 272263
Data columns (total 29 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Date                272264 non-null  object 
 1   Year                272264 non-null  int64  
 2   Month               272264 non-null  int64  
 3   Day                 272264 non-null  int64  
 4   Hour                272264 non-null  int64  
 5   DayOfWeek           272264 non-null  int64  
 6   Season              272264 non-null  object 
 7   Primary Type        272264 non-null  object 
 8   District            272264 non-null  int64  
 9   Community Area      272264 non-null  int64  
 10  Ward                272264 non-null  int64  
 11  Latitude            272264 non-null  float64
 12  Longitude           272264 non-null  float64
 13  Arrest              272264 non-null  bool   
 14  Domestic            272264 non-null  bool   
 15  Weekday          

In [2]:
# Check for missing values
missing_values = df.isnull().sum()

print("\n Missing Values:")
print(missing_values[missing_values > 0])



 Missing Values:
Crime_Count_7D        1
Crime_Count_30D       1
Crime_Count_90D       1
Crime_MA_7D           1
Crime_MA_30D          1
Crime_Diff_7D_30D     1
Crime_Diff_30D_90D    1
dtype: int64


In [3]:
# Filling missing values with 0 for trend columns
trend_cols = [
    'Crime_Count_7D', 'Crime_Count_30D', 'Crime_Count_90D',
    'Crime_MA_7D', 'Crime_MA_30D',
    'Crime_Diff_7D_30D', 'Crime_Diff_30D_90D'
]

df[trend_cols] = df[trend_cols].fillna(0)

# Verify no missing values remain
print("\n✅ Missing Values After Imputation:")
print(df.isnull().sum().sum())  # Should be 0



✅ Missing Values After Imputation:
0


In [4]:
# Check data types
print("\n🔍 Data Types:")
print(df.dtypes)



🔍 Data Types:
Date                   object
Year                    int64
Month                   int64
Day                     int64
Hour                    int64
DayOfWeek               int64
Season                 object
Primary Type           object
District                int64
Community Area          int64
Ward                    int64
Latitude              float64
Longitude             float64
Arrest                   bool
Domestic                 bool
Weekday                 int64
WeekOfYear              int64
Month_sin             float64
Month_cos             float64
Hour_sin              float64
Hour_cos              float64
Crime_Count_7D        float64
Crime_Count_30D       float64
Crime_Count_90D       float64
Crime_MA_7D           float64
Crime_MA_30D          float64
Crime_Diff_7D_30D     float64
Crime_Diff_30D_90D    float64
Crime_Cluster           int64
dtype: object


In [5]:
# One-Hot Encoding for Season
season_encoded = pd.get_dummies(df['Season'], prefix='Season', drop_first=True)
df = pd.concat([df, season_encoded], axis=1)

# Drop the original column
df.drop('Season', axis=1, inplace=True)

print("\n✅ One-Hot Encoding Applied for Season:")
print(df.columns)



✅ One-Hot Encoding Applied for Season:
Index(['Date', 'Year', 'Month', 'Day', 'Hour', 'DayOfWeek', 'Primary Type',
       'District', 'Community Area', 'Ward', 'Latitude', 'Longitude', 'Arrest',
       'Domestic', 'Weekday', 'WeekOfYear', 'Month_sin', 'Month_cos',
       'Hour_sin', 'Hour_cos', 'Crime_Count_7D', 'Crime_Count_30D',
       'Crime_Count_90D', 'Crime_MA_7D', 'Crime_MA_30D', 'Crime_Diff_7D_30D',
       'Crime_Diff_30D_90D', 'Crime_Cluster', 'Season_Spring', 'Season_Summer',
       'Season_Winter'],
      dtype='object')


In [6]:
from sklearn.preprocessing import LabelEncoder

# ✅ Step 1: Apply Label Encoding
le = LabelEncoder()

# Create a new column for encoded labels
df['Primary_Type_Encoded'] = le.fit_transform(df['Primary Type'])

# ✅ Display the mapping
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("\n✅ Label Encoding Applied for Primary Type:")
print(label_mapping)



✅ Label Encoding Applied for Primary Type:
{'ARSON': np.int64(0), 'ASSAULT': np.int64(1), 'BATTERY': np.int64(2), 'BURGLARY': np.int64(3), 'CONCEALED CARRY LICENSE VIOLATION': np.int64(4), 'CRIM SEXUAL ASSAULT': np.int64(5), 'CRIMINAL DAMAGE': np.int64(6), 'CRIMINAL SEXUAL ASSAULT': np.int64(7), 'CRIMINAL TRESPASS': np.int64(8), 'DECEPTIVE PRACTICE': np.int64(9), 'DOMESTIC VIOLENCE': np.int64(10), 'GAMBLING': np.int64(11), 'HOMICIDE': np.int64(12), 'HUMAN TRAFFICKING': np.int64(13), 'INTERFERENCE WITH PUBLIC OFFICER': np.int64(14), 'INTIMIDATION': np.int64(15), 'KIDNAPPING': np.int64(16), 'LIQUOR LAW VIOLATION': np.int64(17), 'MOTOR VEHICLE THEFT': np.int64(18), 'NARCOTICS': np.int64(19), 'NON - CRIMINAL': np.int64(20), 'NON-CRIMINAL': np.int64(21), 'NON-CRIMINAL (SUBJECT SPECIFIED)': np.int64(22), 'OBSCENITY': np.int64(23), 'OFFENSE INVOLVING CHILDREN': np.int64(24), 'OTHER NARCOTIC VIOLATION': np.int64(25), 'OTHER OFFENSE': np.int64(26), 'PROSTITUTION': np.int64(27), 'PUBLIC INDECEN

In [7]:
# ✅ Convert boolean features to 0/1
df['Arrest'] = df['Arrest'].astype(int)
df['Domestic'] = df['Domestic'].astype(int)

print("\n✅ Boolean Encoding Applied for Arrest & Domestic:")
print(df[['Arrest', 'Domestic']].head())



✅ Boolean Encoding Applied for Arrest & Domestic:
   Arrest  Domestic
0       0         1
1       0         0
2       0         0
3       0         1
4       0         0


In [8]:
# ✅ Identify Rare Crime Types (fewer than 5 occurrences)
rare_threshold = 5
rare_types = df['Primary_Type_Encoded'].value_counts()
rare_crimes = rare_types[rare_types < rare_threshold].index

print("\n⚠️ Rare Crime Types (Less than 5 occurrences):")
print(rare_crimes)

# ✅ Merge Rare Crimes into a Single Category
df['Primary_Type_Encoded'] = df['Primary_Type_Encoded'].apply(lambda x: -1 if x in rare_crimes else x)

print("\n✅ Merged Rare Crimes into 'Rare Crime' Category")
print(df['Primary_Type_Encoded'].value_counts())



⚠️ Rare Crime Types (Less than 5 occurrences):
Index([10], dtype='int64', name='Primary_Type_Encoded')

✅ Merged Rare Crimes into 'Rare Crime' Category
Primary_Type_Encoded
 34    52318
 2     45860
 6     28510
 19    23718
 1     16242
 26    15207
 3     13494
 18    11980
 9     10376
 31     9266
 8      6688
 35     3931
 27     3045
 29     2673
 24     2605
 32     2535
 14     2507
 0      2457
 12     2391
 33     2383
 17     2377
 15     2289
 16     2261
 5      2156
 11     1810
 7      1244
 23      750
 4       532
 28      196
 21      158
 25      144
 13       91
 20       37
 30       23
 22        9
-1         1
Name: count, dtype: int64


In [9]:
# Identify low-occurrence crime types (fewer than 50 occurrences)
low_occurrence_crimes = df['Primary_Type_Encoded'].value_counts()
small_classes = low_occurrence_crimes[low_occurrence_crimes < 50]

print("\n⚠️ Small Crime Categories (Occurrences < 50):")
print(small_classes)



⚠️ Small Crime Categories (Occurrences < 50):
Primary_Type_Encoded
 20    37
 30    23
 22     9
-1      1
Name: count, dtype: int64


In [10]:
# Merge rare crime (-1) with label 22
merge_target = 22  # Target small crime label
df['Primary_Type_Encoded'] = df['Primary_Type_Encoded'].replace(-1, merge_target)

print("\n✅ Rare Crime Merged with Label 22")
print(df['Primary_Type_Encoded'].value_counts())



✅ Rare Crime Merged with Label 22
Primary_Type_Encoded
34    52318
2     45860
6     28510
19    23718
1     16242
26    15207
3     13494
18    11980
9     10376
31     9266
8      6688
35     3931
27     3045
29     2673
24     2605
32     2535
14     2507
0      2457
12     2391
33     2383
17     2377
15     2289
16     2261
5      2156
11     1810
7      1244
23      750
4       532
28      196
21      158
25      144
13       91
20       37
30       23
22       10
Name: count, dtype: int64


In [11]:
from sklearn.model_selection import train_test_split

# 🎯 Splitting features and target
X = df.drop(['Crime_Cluster'], axis=1)   # Features
y = df['Crime_Cluster']                   # Target

# ⚙️ Stratified train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\n✅ Stratified Split Done:")
print(f"Training Set: {X_train.shape}")
print(f"Test Set: {X_test.shape}")



✅ Stratified Split Done:
Training Set: (217811, 31)
Test Set: (54453, 31)


In [12]:
print("\n🔍 Class Distribution in Train Set:")
print(y_train.value_counts(normalize=True))

print("\n🔍 Class Distribution in Test Set:")
print(y_test.value_counts(normalize=True))



🔍 Class Distribution in Train Set:
Crime_Cluster
1    0.156856
5    0.139162
8    0.108319
4    0.105936
9    0.092048
3    0.091497
6    0.091281
7    0.085882
0    0.084353
2    0.044667
Name: proportion, dtype: float64

🔍 Class Distribution in Test Set:
Crime_Cluster
1    0.156851
5    0.139166
8    0.108314
4    0.105945
9    0.092043
3    0.091510
6    0.091271
7    0.085891
0    0.084348
2    0.044662
Name: proportion, dtype: float64


In [13]:
from sklearn.preprocessing import MinMaxScaler

# Select numerical columns for scaling (excluding target variable)
scale_cols = [
    'Crime_Count_7D', 'Crime_Count_30D', 'Crime_Count_90D',
    'Crime_MA_7D', 'Crime_MA_30D',
    'Crime_Diff_7D_30D', 'Crime_Diff_30D_90D'
]

# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Scale train and test sets
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print("\n✅ MinMax Scaling Applied to Numerical Features!")



✅ MinMax Scaling Applied to Numerical Features!


In [14]:
# Verify scaling range (should be between 0 and 1)
print("\n🔍 Scaled Train Set Ranges:")
print(X_train[scale_cols].describe())

print("\n🔍 Scaled Test Set Ranges:")
print(X_test[scale_cols].describe())



🔍 Scaled Train Set Ranges:
       Crime_Count_7D  Crime_Count_30D  Crime_Count_90D    Crime_MA_7D  \
count   217811.000000    217811.000000    217811.000000  217811.000000   
mean         0.634154         0.703773         0.731990       0.659153   
std          0.154850         0.166711         0.174677       0.159549   
min          0.000000         0.000000         0.000000       0.000000   
25%          0.500000         0.555481         0.572312       0.519413   
50%          0.646154         0.723262         0.758806       0.672912   
75%          0.766667         0.854947         0.886634       0.799608   
max          1.000000         1.000000         1.000000       1.000000   

        Crime_MA_30D  Crime_Diff_7D_30D  Crime_Diff_30D_90D  
count  217811.000000      217811.000000       217811.000000  
mean        0.716353           0.310929            0.274209  
std         0.170767           0.166086            0.180452  
min         0.000000           0.000000            0.0000

In [15]:

# Save to CSV
X_train.to_csv("x_train.csv", index=False)
X_test.to_csv("x_test.csv", index=False)

print("\n✅ Final Datasets Exported Successfully!")
print("🔹 Train Dataset → x_train.csv")
print("🔹 Test Dataset  → x_test.csv")



✅ Final Datasets Exported Successfully!
🔹 Train Dataset → x_train.csv
🔹 Test Dataset  → x_test.csv


In [16]:
# Export y_train
y_train.to_csv("y_train.csv", index=False)

# Export y_test
y_test.to_csv("y_test.csv", index=False)


In [17]:
# 🛠️ Create Regression Datasets

# 1️⃣ Remove trend-based features from X_train and X_test
X_train_reg = X_train.drop(['Crime_Count_7D', 'Crime_Count_30D', 'Crime_Count_90D'], axis=1)
X_test_reg = X_test.drop(['Crime_Count_7D', 'Crime_Count_30D', 'Crime_Count_90D'], axis=1)

# 2️⃣ Map Crime_Count_30D as the target for regression using `df`
y_train_reg = df.loc[X_train.index, "Crime_Count_30D"].values
y_test_reg = df.loc[X_test.index, "Crime_Count_30D"].values

# ✅ Verify Shapes
print("\n🔥 Regression Datasets Created:")
print(f"X_train_reg shape: {X_train_reg.shape}")
print(f"X_test_reg shape: {X_test_reg.shape}")
print(f"y_train_reg shape: {y_train_reg.shape}")
print(f"y_test_reg shape: {y_test_reg.shape}")



🔥 Regression Datasets Created:
X_train_reg shape: (217811, 28)
X_test_reg shape: (54453, 28)
y_train_reg shape: (217811,)
y_test_reg shape: (54453,)


In [19]:
import pandas as pd

# Convert y arrays to Pandas Series
y_train_reg_series = pd.Series(y_train_reg, name="Crime_Count_30D")
y_test_reg_series = pd.Series(y_test_reg, name="Crime_Count_30D")

# Save the regression datasets to CSV
X_train_reg.to_csv("X_train_reg.csv", index=False)
X_test_reg.to_csv("X_test_reg.csv", index=False)
y_train_reg_series.to_csv("y_train_reg.csv", index=False)
y_test_reg_series.to_csv("y_test_reg.csv", index=False)

print("\n✅ Regression datasets saved successfully!")



✅ Regression datasets saved successfully!


In [21]:
import numpy as np
# 🔥 Check min, max, and basic stats
print("\n📊 Y_train_reg Range:")
print(f"Min: {np.min(y_train_reg)}")
print(f"Max: {np.max(y_train_reg)}")
print(f"Mean: {np.mean(y_train_reg)}")
print(f"Std Dev: {np.std(y_train_reg)}")

print("\n📊 Y_test_reg Range:")
print(f"Min: {np.min(y_test_reg)}")
print(f"Max: {np.max(y_test_reg)}")
print(f"Mean: {np.mean(y_test_reg)}")
print(f"Std Dev: {np.std(y_test_reg)}")


📊 Y_train_reg Range:
Min: 0.0
Max: 1496.0
Mean: 1052.8439610487992
Std Dev: 249.39967819449225

📊 Y_test_reg Range:
Min: 4.0
Max: 1493.0
Mean: 1051.5853488329385
Std Dev: 248.88556656348803
